# NB_SEC_Notify · Phase 5 · sends queued mails through Microsoft Graph
Order in PL_SEC_Lifecycle: Lifecycle → Sync → **Notify**. Sends from CONFIG["mail_sender"] with the app's Mail.Send permission
(restrict that permission to this one mailbox with Exchange RBAC for Applications). Failed mails are retried for 2 days (NOT-03).

In [ ]:
%run NB_SEC_Common

In [ ]:
q = sql_df("""
SELECT n.NotificationId, n.NotificationType, n.SentTo, n.Cc, n.Status, r.RequestNumber, r.TicketRef, r.StartDate, r.EndDate, r.ScopeValue,
       r.StatusReason, u.DisplayName, rt.DisplayName AS RoleName, rt.RoleCode, d.DomainName, sv.Label AS ScopeLabel
FROM sec.NotificationLog n JOIN sec.AccessRequest r ON r.RequestId = n.RequestId JOIN sec.UserProfile u ON u.UserId = r.UserId
JOIN sec.RoleTemplate rt ON rt.RoleTemplateId = r.RoleTemplateId JOIN sec.Domain d ON d.DomainId = rt.DomainId
JOIN sec.ScopeValue sv ON sv.ScopeValue = r.ScopeValue
WHERE n.ScheduledFor <= SYSUTCDATETIME() AND (n.Status = 'Queued' OR (n.Status = 'Failed' AND n.ScheduledFor >= DATEADD(DAY, -2, SYSUTCDATETIME())))""")

SUBJECT = {
    "APPROVAL_REQUEST": "Approval needed: {RoleName} for {DisplayName} ({RequestNumber})",
    "APPROVED": "Access approved: {RoleName}, starts {StartDate} ({RequestNumber})",
    "REJECTED": "Access request rejected ({RequestNumber})",
    "ENABLED": "Access enabled: {RoleName}, {ScopeLabel} ({RequestNumber})",
    "EXPIRY_T7": "Your {RoleName} access ends in 7 days ({RequestNumber})",
    "EXPIRY_T1": "Your {RoleName} access ends tomorrow ({RequestNumber})",
    "EXTENDED": "Access extended to {EndDate} ({RequestNumber})",
    "REVOKED": "Access removed: {RoleName}, {ScopeLabel} ({RequestNumber})",
}
LEAD = {
    "APPROVAL_REQUEST": "A data access request is waiting for your decision in the service portal.",
    "APPROVED": "The request has been approved. Access will be switched on automatically on the start date.",
    "REJECTED": "The request was not approved. Reason: {StatusReason}.",
    "ENABLED": "Access is now active. You can use it in the SQL analytics endpoint, Power BI, Copilot and the data agents.",
    "EXPIRY_T7": "This access ends on {EndDate}. If it is still needed, raise an Extend request in the service portal.",
    "EXPIRY_T1": "This access ends tomorrow, {EndDate}. Raise an Extend request today if it is still needed.",
    "EXTENDED": "The end date has been moved to {EndDate}.",
    "REVOKED": "This access has been removed. Reason: {StatusReason}.",
}
def body(x):
    v = {k: ("" if pd.isna(val) else val) for k, val in x._asdict().items()}
    rows = "".join(f"<tr><td style='padding:4px 12px 4px 0;color:#606167'>{k}</td><td>{val}</td></tr>" for k, val in
                   [("Person", v["DisplayName"]), ("Domain", v["DomainName"]), ("Role", f"{v['RoleName']} ({v['RoleCode']})"),
                    ("Operating unit", v["ScopeLabel"]), ("Valid", f"{v['StartDate']} to {v['EndDate']}"), ("Request", v["RequestNumber"]), ("Ticket", v["TicketRef"])])
    html = (f"<div style='font-family:Segoe UI,Arial,sans-serif;font-size:14px;color:#080b12'><p>{LEAD[v['NotificationType']].format(**v)}</p>"
            f"<table style='border-collapse:collapse;font-size:14px'>{rows}</table>"
            "<p style='color:#606167;font-size:12px'>KDD Data Platform · OneLake access service. Automated message; replies are not monitored.</p></div>")
    return SUBJECT[v["NotificationType"]].format(**v), html

sent = failed = 0
for x in q.itertuples():
    subject, html = body(x)
    msg = {"message": {"subject": subject, "body": {"contentType": "HTML", "content": html},
                       "toRecipients": [{"emailAddress": {"address": a}} for a in str(x.SentTo).split(";") if a],
                       "ccRecipients": [{"emailAddress": {"address": a}} for a in str(x.Cc or "").split(";") if a]},
           "saveToSentItems": True}
    r = call("graph", "POST", f"/users/{CONFIG['mail_sender']}/sendMail", ok=None, json=msg)
    if r.status_code == 202:
        sql_exec("UPDATE sec.NotificationLog SET Status = 'Sent', Error = NULL WHERE NotificationId = ?", (int(x.NotificationId),)); sent += 1
    else:
        sql_exec("UPDATE sec.NotificationLog SET Status = 'Failed', Error = ? WHERE NotificationId = ?", (r.text[:300], int(x.NotificationId))); failed += 1
print(f"Sent {sent}, failed {failed}")